# 12 — Explain the pinned classifier and publish its dashboard inputs

Run 11 first, then Restart Kernel and Run All. This notebook uses the frozen candidate from 10/11, its own cutoff, validation results and food-group evidence. It never fits or retunes the classifier. No legacy combined-training checkpoint is required.

Verify model/input hashes and versions → score the prepared latest week → join the separate anomaly signal → compute bounded SHAP batches → verify additivity and score agreement → publish a separate deployment pointer last. The old dashboard export stays available if any check fails.

The selected model's validation-cutoff metrics are apparent selection-period performance, not independent test results. Reference SHAP uses a fixed sample of original 2025 inputs, not the entire 2024–2025 training population.

To expand explanations for the current saved snapshot, run main Notebook 12 with its default `EXPORT_MODE = "auto"` using Restart Kernel → Run All. It preserves the saved prices and model scores and verifies full latest-row coverage. Main 13 is needed only for newer observations.

## 1. Verify the frozen deployment

Use the exact candidate pinned from 10 and the detector linked by 11. Verify model hashes, package versions, selected cutoff and candidate-specific validation evidence before scoring.

In [1]:
import hashlib
import json
from importlib import metadata, reload
from pathlib import Path
from time import perf_counter
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
import pricecatcher_config as cfg
cfg = reload(cfg)
REPOSITORY = Path(cfg.__file__).resolve().parent
EXPORT_MODE = "auto"  # auto, saved_snapshot, or original_export
pin, STUDY_DIR, study = cfg.dashboard_candidate()
selection = study["selection"]
PREPARED_DIR = cfg.PROJECT_ROOT / "experiments" / "holdout" / "05_modelling"
DASHBOARD_DIR = cfg.PROJECT_ROOT / "08_dashboard" / "deployments" / pin["deployment_id"]
KEYS = ["item_code", "state", "district", "premise_type", "week_start"]
print("Pinned candidate:", pin["candidate"], "| Cutoff:", selection["score_cutoff"])
print("Separate deployment export:", DASHBOARD_DIR)

Pinned candidate: Longer-history LightGBM (tuned) | Cutoff: 0.22099133987116373
Separate deployment export: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/08_dashboard/deployments/8817e80ce245dedd


## Small reusable calculations
Use the exact saved preprocessing: categorical values are converted to strings as in 06, and numeric missing values are handled by the frozen imputer. Expanded one-hot and missing-indicator contributions are **summed** back to original feature names; this display grouping preserves additivity but does not create a causal interpretation or a new jointly grouped SHAP game.

In [2]:
def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def model_inputs(frame, settings):
    result = frame[settings["feature_columns"]].copy()
    for name in settings["categorical_features"]:
        result[name] = result[name].astype("string").fillna("__missing__").astype(str)
    return result


def dense_matrix(matrix):
    # Keep the offline explanation batch modest even if category counts grow.
    if matrix.shape[0] * matrix.shape[1] > 2_000_000:
        raise ValueError("Explanation matrix is too large for this bounded cache. Review category counts before changing the protocol.")
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


def source_features(preprocess, settings):
    numeric = preprocess.named_transformers_["numeric"]
    if hasattr(numeric, "named_steps"):
        numeric = numeric.named_steps["impute"]
    names = [str(name).removeprefix("missingindicator_")
             for name in numeric.get_feature_names_out(settings["numeric_features"])]
    encoder = preprocess.named_transformers_["categorical"]
    if encoder.drop is not None or encoder.min_frequency is not None or encoder.max_categories is not None:
        raise ValueError("This cache expects the declared one-hot encoder.")
    for name, categories in zip(settings["categorical_features"], encoder.categories_):
        names.extend([name] * len(categories))
    if not set(names).issubset(settings["feature_columns"]):
        raise ValueError("Cannot map the frozen preprocessing back to original features.")
    return names


def explain_rows(pipeline, family, frame, background, settings, explainer=None):
    """Return grouped SHAP, its baseline, raw output and the unchanged model score."""
    features = settings["feature_columns"]
    if frame.empty:
        return np.empty((0, len(features))), np.array([]), np.array([]), np.array([]), explainer
    preprocess, estimator = pipeline.named_steps["preprocess"], pipeline.named_steps["model"]
    width = len(source_features(preprocess, settings))
    if width < 1:
        raise ValueError("No transformed features are available for explanation.")
    batch_rows = min(128, max(1, 2_000_000 // width))
    if len(frame) > batch_rows:
        parts = []
        for start in range(0, len(frame), batch_rows):
            values, base, raw, scores, explainer = explain_rows(
                pipeline, family, frame.iloc[start:start + batch_rows], background, settings, explainer)
            parts.append((values, base, raw, scores))
            print(f"Explained {min(start + batch_rows, len(frame)):,} / {len(frame):,} rows", flush=True)
        return (*[np.concatenate([part[i] for part in parts], axis=0) for i in range(4)], explainer)
    X = model_inputs(frame, settings)
    matrix = preprocess.transform(X)
    names = source_features(preprocess, settings)
    if len(names) != matrix.shape[1] or list(estimator.classes_) != [0, 1]:
        raise ValueError("Expected the frozen binary 0/1 model and matching transformed features.")
    if family == "XGBoost":
        # Native TreeSHAP preserves CSR semantics used during XGBoost training in 08.
        import xgboost as xgb
        contributions = estimator.get_booster().predict(xgb.DMatrix(matrix), pred_contribs=True)
        values, base = contributions[:, :-1], contributions[:, -1]
        raw = estimator.predict(matrix, output_margin=True)
    else:
        import shap
        matrix = dense_matrix(matrix)
        if explainer is None:
            if family == "Logistic Regression":
                reference = dense_matrix(preprocess.transform(model_inputs(background, settings)))
                explainer = shap.LinearExplainer(estimator, reference)
            elif family in {"LightGBM", "Random Forest"}:
                explainer = shap.TreeExplainer(estimator, feature_perturbation="tree_path_dependent", model_output="raw")
            else:
                raise ValueError(f"Unsupported frozen model family: {family}")
        result = explainer(matrix)
        values, base = np.asarray(result.values), np.asarray(result.base_values)
        if values.ndim == 3:
            values = values[:, :, 1]
            base = base[:, 1] if base.ndim == 2 else base[1]
        base = np.broadcast_to(np.squeeze(base), (len(frame),)).copy()
        if family == "Logistic Regression":
            raw = estimator.decision_function(matrix)
        elif family == "LightGBM":
            raw = estimator.booster_.predict(matrix, raw_score=True)
        else:
            raw = estimator.predict_proba(matrix)[:, 1]
    grouped = np.column_stack([values[:, np.asarray(names) == name].sum(axis=1) for name in features])
    scores = pipeline.predict_proba(X)[:, 1]
    np.testing.assert_allclose(base + grouped.sum(axis=1), raw, rtol=1e-5, atol=1e-5)
    reconstructed = raw if family == "Random Forest" else expit(raw)
    np.testing.assert_allclose(reconstructed, scores, rtol=1e-5, atol=1e-6)
    return grouped, np.asarray(base), np.asarray(raw), scores, explainer


def anomaly_inputs(frame):
    """The six fixed version-1 formulas from 11; this does not fit a detector."""
    price = frame.weekly_median_price
    return pd.DataFrame({
        "current_change_pct": frame.weekly_price_change_pct,
        "change_2w_pct": 100 * (price / frame.lag_2w_price - 1),
        "change_4w_pct": 100 * (price / frame.lag_4w_price - 1),
        "deviation_4w_pct": 100 * (price / frame.rolling_mean_4w - 1),
        "within_week_variation_pct": 100 * frame.weekly_std_price / price,
        "history_variation_4w_pct": 100 * frame.rolling_std_4w / frame.rolling_mean_4w,
    }, index=frame.index)


def match_latest(frame, latest, name):
    """Align saved results to exactly the prepared latest series; never substitute old weeks."""
    if frame.duplicated(KEYS).any() or len(frame) != len(latest):
        raise ValueError(f"{name}: duplicate rows or different latest coverage. Rerun the matching completed stage.")
    aligned = latest[KEYS].merge(frame, on=KEYS, how="left", validate="one_to_one", indicator=True)
    if not aligned["_merge"].eq("both").all():
        raise ValueError(f"{name}: stale or mismatched latest rows.")
    return aligned.drop(columns="_merge")

## Choose the export route automatically

Leave `EXPORT_MODE = "auto"` in the setup cell. If a dashboard snapshot already belongs to this pin, Run All expands/verifies **that same saved week**, including an existing refresh. No price downloads or repricing take place. A first export, or a different newly pinned deployment, follows the original export steps below.

Advanced override: `saved_snapshot` requires an existing snapshot and stops if its model differs; `original_export` builds the original prepared-week export. Earlier caches remain preserved. The reference summary still uses 200 historical rows; **every supported latest row** is explained in bounded batches.

In [3]:
def select_export_mode(requested, run, pin):
    """Keep an existing matching snapshot; build a base export only when needed."""
    import app
    if requested not in {"auto", "saved_snapshot", "original_export"}:
        raise ValueError("EXPORT_MODE must be auto, saved_snapshot, or original_export.")
    if requested == "original_export":
        return requested
    entry_path = app.dashboard_entry(run)
    if not entry_path.is_file():
        if requested == "saved_snapshot":
            raise FileNotFoundError("No saved dashboard snapshot exists. Complete 11 and use original_export for the first export.")
        return "original_export"
    entry = app.read_json(entry_path)
    if requested == "saved_snapshot" or entry["forecast_model"].get("deployment_id") == pin["deployment_id"]:
        return "saved_snapshot"
    return "original_export"


def expand_shap_cache():
    """Explain every row of the already published snapshot; never refresh prices."""
    import ast
    import shutil
    from time import perf_counter
    import joblib
    import numpy as np
    import pandas as pd
    from scipy.special import expit
    import app

    run = app.active_run().resolve()
    pin_path = run / "11_deployment" / "deployment_selection.json"
    if not pin_path.is_file():
        raise FileNotFoundError("Restore the existing deployment pin; expansion never creates one.")
    import pricecatcher_config as cfg
    if cfg.PROJECT_ROOT.resolve() != run:
        raise ValueError("Dashboard and configuration identify different active runs.")
    pin, study_dir, study = cfg.dashboard_candidate()
    entry_path = app.dashboard_entry(run)
    previous_entry = entry_path.read_bytes()
    entry, manifest, latest, explanations, _, _ = app.load_exports(str(run), app.export_signature(run))
    source_cache = app.inside(entry_path.parent, entry["cache_dir"])
    for name, expected in manifest["output_sha256"].items():
        if app.file_hash(app.inside(source_cache, name)) != expected:
            raise ValueError("A saved explanation-cache artifact changed.")
    model = entry["forecast_model"]
    if model.get("deployment_id") != pin["deployment_id"] or model["score_cutoff"] != pin["score_cutoff"]:
        raise ValueError("Publish the pinned deployment before expanding its explanations.")
    model_path = app.inside(run, model["model_path"])
    if model_path.parent != study_dir.resolve() or app.file_hash(model_path) != model["sha256"]:
        raise ValueError("The published classifier differs from the pinned study.")
    for record in [study, manifest]:
        for package, expected in record.get("package_versions", {}).items():
            if cfg.installed_package_version(package) != expected:
                raise ValueError(f"Use the saved {package} version before expanding explanations.")
    local = explanations.loc[explanations.cohort.eq("latest")]
    if (local.explanation_id.is_unique and set(local.explanation_id) == set(latest.forecast_id)
            and latest.has_cached_explanation.all()):
        print(f"All {len(latest):,} latest rows already have verified explanations. Nothing changed.")
        return entry_path
    settings_path = run / "experiments" / "holdout" / "05_modelling" / "experiment_settings.json"
    settings = json.loads(settings_path.read_text())
    if settings["feature_columns"] != study["selection"]["feature_columns"]:
        raise ValueError("Published model features differ from the frozen study.")
    protected = {pin_path: app.file_hash(pin_path), model_path: model["sha256"],
                 study_dir / "study_manifest.json": pin["study_manifest_sha256"],
                 settings_path: study["input_sha256"][str(settings_path.relative_to(run))]}
    source_cache = app.inside(entry_path.parent, entry["cache_dir"])
    source_manifest = source_cache / "explanation_manifest.json"
    protected[source_manifest] = entry["cache_manifest_sha256"]
    files = app.TABLES + (["refresh_history.parquet"] if "refresh" in entry else [])
    for name in files:
        protected[source_cache / name] = manifest["output_sha256"][name]
    notebook = json.loads((REPOSITORY / "12_shap_and_dashboard_inputs.ipynb").read_text())
    names = {"model_inputs", "dense_matrix", "source_features", "explain_rows"}
    functions = [node for cell in notebook["cells"] if cell["cell_type"] == "code"
                 for node in ast.parse("".join(cell["source"])).body
                 if isinstance(node, ast.FunctionDef) and node.name in names]
    if sorted(node.name for node in functions) != sorted(names):
        raise ValueError("Missing or duplicate explanation calculations.")
    namespace = {"np": np, "pd": pd, "expit": expit}
    exec(compile(ast.Module(body=functions, type_ignores=[]), "explanation calculations only", "exec"), namespace)
    protocol = {**manifest["protocol"], "schema_version": 3, "latest_rows": "all",
                "maximum_batch_rows": 128, "maximum_dense_cells": 2_000_000}
    code = "\n".join("".join(cell["source"]) for cell in notebook["cells"] if cell["cell_type"] == "code")
    protocol["calculation_sha256"] = hashlib.sha256(code.encode()).hexdigest()
    family = study["selection"]["family"]
    background = pd.DataFrame(columns=settings["feature_columns"])
    if family == "Logistic Regression":
        train_path = settings_path.parent / "train.parquet"
        protected[train_path] = study["input_sha256"][str(train_path.relative_to(run))]
        if app.file_hash(train_path) != protected[train_path]:
            raise ValueError("Frozen training reference changed.")
        training = pd.read_parquet(train_path, columns=settings["feature_columns"])
        background = training.sample(min(protocol.get("background_rows", 100), len(training)),
                                     random_state=protocol.get("seed", 42))
    started = perf_counter()
    pipeline = joblib.load(model_path)
    values, base, raw, scores, _ = namespace["explain_rows"](pipeline, family, latest, background, settings)
    np.testing.assert_allclose(scores, latest.model_score, rtol=1e-6, atol=1e-8)
    np.testing.assert_array_equal(scores >= pin["score_cutoff"], latest.predicted_surge)
    identifiers = list(dict.fromkeys(cfg.GROUP_KEYS + ["week_start", "next_week_end"]))
    full = latest[identifiers].reset_index(drop=True).copy()
    full["cohort"], full["explanation_id"] = "latest", latest.forecast_id.to_numpy()
    full["base_value"], full["raw_output"], full["model_score"] = base, raw, scores
    shap_columns = ["shap__" + name for name in settings["feature_columns"]]
    full[shap_columns] = values
    combined = pd.concat([explanations.loc[explanations.cohort.eq("historical_reference")], full], ignore_index=True)
    if not combined.explanation_id.is_unique or not np.isfinite(combined[shap_columns + ["base_value", "raw_output", "model_score"]].to_numpy(dtype=float)).all():
        raise ValueError("Invalid full-coverage explanations; previous export preserved.")
    np.testing.assert_allclose(full.base_value + full[shap_columns].sum(axis=1), full.raw_output, rtol=1e-5, atol=1e-5)
    updated = latest.copy()
    updated["has_cached_explanation"] = True
    cache_id = hashlib.sha256(json.dumps({"source": entry["cache_manifest_sha256"], "protocol": protocol}, sort_keys=True).encode()).hexdigest()[:16]
    target = entry_path.parent / f"full_shap_{cache_id}"
    if target.exists():
        raise ValueError(f"A full-SHAP cache already exists at {target}; preserve and inspect it. Nothing overwritten.")
    def check_inputs():
        for path, expected in protected.items():
            if app.file_hash(path) != expected:
                raise ValueError(f"Frozen input changed: {path}. Previous export preserved.")
        if entry_path.read_bytes() != previous_entry:
            raise ValueError("Dashboard publication changed during expansion.")
    check_inputs()
    target.mkdir()
    updated.to_parquet(target / "latest_dashboard.parquet", index=False)
    combined.to_parquet(target / "shap_explanations.parquet", index=False)
    for name in files:
        if name not in {"latest_dashboard.parquet", "shap_explanations.parquet"}:
            shutil.copyfile(source_cache / name, target / name)
    (target / "previous_entry.json").write_bytes(previous_entry)
    output_hashes = {name: app.file_hash(target / name) for name in files + ["previous_entry.json"]}
    expanded = {**manifest, "protocol": protocol, "latest_explained_rows": len(latest),
                "offline_cache_build_seconds": perf_counter() - started, "output_sha256": output_hashes,
                "coverage_expansion": {"previous_entry": str((target / "previous_entry.json").relative_to(run)),
                                       "source_cache_manifest_sha256": entry["cache_manifest_sha256"],
                                       "generated_at": pd.Timestamp.now(tz="Asia/Kuala_Lumpur").isoformat(),
                                       "prices_refreshed": False, "model_fitted": False}}
    expanded["limitations"] = [value for value in manifest.get("limitations", [])
                               if value not in {"Prioritised latest subset", "All supported latest rows explained; historical reference remains sampled"}] + ["All supported latest rows explained; historical reference remains sampled"]
    (target / "explanation_manifest.json").write_text(json.dumps(expanded, indent=2))
    published = {**entry, "cache_dir": target.name,
                 "cache_manifest_sha256": app.file_hash(target / "explanation_manifest.json"),
                 "limitations": expanded["limitations"],
                 "files": {name: str(Path(target.name) / name) for name in files}}
    # app.load_exports adds display-only audit fields; retain the original publication schema.
    for key in ["refresh_audit", "original_supported_rows"]:
        if key not in json.loads(previous_entry):
            published.pop(key, None)
    check_inputs()
    temporary = entry_path.with_suffix(".full-shap.tmp")
    temporary.write_text(json.dumps(published, indent=2))
    temporary.replace(entry_path)
    try:
        app.load_exports(str(run), app.export_signature(run))
    except Exception:
        temporary.write_bytes(previous_entry)
        temporary.replace(entry_path)
        raise
    print(f"Published full SHAP coverage: {len(latest):,} / {len(latest):,} latest rows.")
    print("Same observation week, model, scores, cutoff and anomaly flags. No source downloads or test-label reads.")
    print("Full SHAP cache:", target)
    print("Previous cache retained; previous entry archived at:", target / "previous_entry.json")
    return entry_path


export_mode = select_export_mode(EXPORT_MODE, cfg.PROJECT_ROOT, pin)
print("Export route:", export_mode)


2026-11-16 18:20:09.438 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Export route: saved_snapshot


In [4]:
if export_mode == "original_export":
    settings = json.loads((PREPARED_DIR / "experiment_settings.json").read_text())
    FEATURES = settings["feature_columns"]
    IDENTIFIERS = cfg.GROUP_KEYS + ["week_start", "next_week_end"]
    COLUMNS = list(dict.fromkeys(IDENTIFIERS + FEATURES))
    if FEATURES != selection["feature_columns"]:
        raise ValueError("Selected model and prepared feature lists differ.")
    for package, expected in study["package_versions"].items():
        if cfg.installed_package_version(package) != expected:
            raise ValueError(f"Use the selected model's {package} version.")
    family = selection["family"]
    model_name = {"LightGBM":"lightgbm.joblib", "Random Forest":"random_forest.joblib",
                  "XGBoost":"xgboost.joblib", "Logistic Regression":"logistic_regression.joblib"}[family]
    forecast_model_path = STUDY_DIR / model_name
    forecast_sha256 = file_hash(forecast_model_path)
    forecast_model = joblib.load(forecast_model_path)
    forecast_version = "deployment_" + pin["deployment_id"]
    units = "uncalibrated model-score units" if family == "Random Forest" else "model log-odds"
    # Locate the detector by its exact training identity, independently of classifier identity.
    train_sha = study["input_sha256"]["experiments/holdout/05_modelling/train.parquet"]
    link_path = cfg.PROJECT_ROOT / "11_deployment" / "anomaly_selection.json"
    if not link_path.is_file():
        raise FileNotFoundError("Run 11 first to verify the separate detector and publish its scores.")
    link = json.loads(link_path.read_text())
    if link["deployment_id"] != pin["deployment_id"] or link["training_input_sha256"] != train_sha:
        raise ValueError("Run 11 for this frozen deployment candidate.")
    ANOMALY_DIR = (cfg.PROJECT_ROOT / link["manifest_path"]).parent
    if file_hash(ANOMALY_DIR / "anomaly_manifest.json") != link["manifest_sha256"] or file_hash(ANOMALY_DIR / "anomaly_scores.parquet") != link["scores_sha256"]:
        raise ValueError("Anomaly outputs changed after stage 11.")
    anomaly_paths = [ANOMALY_DIR / name for name in ["anomaly_manifest.json", "isolation_forest.joblib", "anomaly_scores.parquet"]]
    anomaly_manifest = json.loads(anomaly_paths[0].read_text())
    if file_hash(anomaly_paths[1]) != anomaly_manifest["model_sha256"]:
        raise ValueError("Anomaly detector changed.")
    for package, expected in anomaly_manifest["package_versions"].items():
        if cfg.installed_package_version(package) != expected:
            raise ValueError(f"Use the detector's {package} version.")
    anomaly_model = joblib.load(anomaly_paths[1])
    latest = pd.read_parquet(PREPARED_DIR / "latest_prediction_inputs.parquet", columns=COLUMNS)
    end = pd.Timestamp(settings["evaluation_end"])
    expected_week = end - pd.Timedelta(days=(end.dayofweek + 1) % 7 + 6)
    if latest.empty or latest[KEYS].isna().any().any() or not latest.week_start.eq(expected_week).all():
        raise ValueError("Latest prediction inputs are empty, incomplete or stale.")
    evidence_table = pd.read_csv(STUDY_DIR / "validation_comparison.csv")
    method = selection.get("comparison_method", "Optuna selected")
    evidence = evidence_table.loc[evidence_table.method.eq(method + ": apparent cutoff-selection performance")]
    if len(evidence) != 1:
        raise ValueError("Expected exactly one selected-cutoff validation result.")
    groups = pd.read_csv(STUDY_DIR / "validation_group_metrics.csv")
    group_evidence = json.loads(groups.set_index("project_group").to_json(orient="index", double_precision=15))
    # Freeze the cleaning/source context needed for later refresh, including for 2025-only candidates.
    cleaning_path = PREPARED_DIR.parent / "03_cleaned" / "cleaning_summary.json"
    daily_path = PREPARED_DIR.parent / "03_cleaned" / "daily_prices.parquet"
    cleaning = json.loads(cleaning_path.read_text())
    source = cfg.verify_source_snapshot()
    if cleaning["preparation"] != settings["preparation"] or source["source_snapshot_sha256"] != settings["preparation"].get("source_snapshot_sha256"):
        raise ValueError("Cleaning/source context does not match the prepared model inputs.")
    refresh_inputs = {str(p.relative_to(cfg.PROJECT_ROOT)):file_hash(p) for p in
                      [PREPARED_DIR / "experiment_settings.json", PREPARED_DIR / "train.parquet", cleaning_path, daily_path]}
    forecast_record = {"version": forecast_version, "candidate": pin["candidate"], "display_name": pin["candidate"],
        "family": family, "model_path": str(forecast_model_path.relative_to(cfg.PROJECT_ROOT)), "sha256": forecast_sha256,
        "study_manifest": str((STUDY_DIR / "study_manifest.json").relative_to(cfg.PROJECT_ROOT)),
        "study_manifest_sha256": pin["study_manifest_sha256"], "deployment_id": pin["deployment_id"],
        "training_start": selection.get("training_start", "2025-01"), "training_end": selection["training_end"],
        "score_cutoff": selection["score_cutoff"], "score_interpretation": "uncalibrated model score",
        "evidence_status": "exploratory fold-selected candidate; validation cutoff fitted on the reported period; independent test performance unconfirmed",
        "validation_period": {"after": settings["train_end"], "through": settings["validation_end"]},
        "validation_metrics": json.loads(evidence.to_json(orient="records", double_precision=15))[0], "group_evidence": group_evidence,
        "guidance_reference": "Selected candidate's Jan–Mar 2026 validation; cutoff-selection period, not per-case confidence.",
        "reference_input_scope": "fixed original 2025 training sample; not full longer-history importance",
        "refresh_inputs_sha256": refresh_inputs}
    # Preserve history contracts; only hashes are read for prepared test data here.
    input_hashes = {name.removeprefix("experiments/").replace("/05_modelling/", "/"): value for name,value in study["input_sha256"].items()}
    for name in ["test.parquet", "latest_prediction_inputs.parquet"]:
        input_hashes["holdout/"+name] = file_hash(PREPARED_DIR / name)
    main_paths = [STUDY_DIR / "study_manifest.json", STUDY_DIR / "selection.json", forecast_model_path,
                  STUDY_DIR / "validation_comparison.csv", STUDY_DIR / "validation_group_metrics.csv",
                  cfg.PROJECT_ROOT / "11_deployment" / "deployment_selection.json", link_path]
    print("Selected model and candidate-specific evidence verified.")

## 2. Score the selected classifier and verify anomaly signals

Every supported latest row gets the selected classifier score and frozen cutoff decision. The separate anomaly scores must reproduce the linked 2025 detector. No classifier, imputer, cutoff or detector is refitted here.

In [5]:
if export_mode == "original_export":
    scores = forecast_model.predict_proba(model_inputs(latest, settings))[:, 1]
    anomalies = match_latest(pd.read_parquet(anomaly_paths[2], filters=[("split", "==", "latest")],
        columns=KEYS + ["anomaly_score", "is_anomaly"]), latest, "11 anomaly scores")
    X_anomaly = anomaly_inputs(latest)
    if list(anomaly_manifest["protocol"]["feature_formulas"]) != list(X_anomaly.columns):
        raise ValueError("Anomaly feature definitions differ from the supported version-1 detector.")
    if not np.isclose(-anomaly_model.named_steps["detector"].offset_, anomaly_manifest["training_score_threshold"], rtol=0, atol=1e-12):
        raise ValueError("Anomaly training threshold differs from its saved detector.")
    np.testing.assert_allclose(anomalies.anomaly_score, -anomaly_model.score_samples(X_anomaly), rtol=1e-6, atol=1e-8)
    np.testing.assert_array_equal(anomalies.is_anomaly, anomaly_model.predict(X_anomaly) == -1)
    if not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all() or not np.isfinite(anomalies.anomaly_score).all():
        raise ValueError("Invalid forecast/anomaly scores; old dashboard preserved.")
    dashboard = latest.copy()
    dashboard["forecast_model_version"] = forecast_version
    dashboard["model_score"] = scores
    dashboard["predicted_surge"] = (scores >= selection["score_cutoff"]).astype(int)
    for name in ["anomaly_score", "is_anomaly"]:
        dashboard[name] = anomalies[name].to_numpy()
    dashboard = dashboard.sort_values(["model_score"] + KEYS, ascending=[False] + [True] * len(KEYS)).reset_index(drop=True)
    dashboard["forecast_id"] = [f"latest_{i:05d}" for i in range(len(dashboard))]

## 3. Declare a bounded offline cache
Use up to **200 randomly sampled original 2025 training-input rows** (seed 42) for the descriptive 2025-reference feature summary, and **every supported latest row** for local explanations. Computation is split into batches of at most 128 rows, reduced further to respect the dense-matrix guard. The 200-row historical sample is only for the descriptive global summary, not a cap on local coverage. A linear model uses up to 100 training rows for its independent-feature background. Tree models use TreeSHAP's training path/leaf reference; XGBoost uses its native TreeSHAP to preserve sparse-input semantics.

Do not interpret correlated lags or grouped one-hot columns as independent economic causes. Log-odds contributions are not percentage-point changes in event probability. Random Forest's raw contributions instead reconstruct its uncalibrated positive-class score.

The common 2025 reference describes the chosen model on those inputs; it is not full-population training importance for the longer-history model. A linear model uses this declared reference background.

Only this derived 08 cache may be regenerated if its files/settings change. Frozen models and inputs always remain protected. A complete matching cache is loaded without recomputing SHAP.

In [6]:
if export_mode == "original_export":
    try:
        shap_version = metadata.version("shap")
    except metadata.PackageNotFoundError:
        raise ModuleNotFoundError("SHAP is missing. Run %pip install -r requirements.txt in a temporary notebook cell, then restart and Run All.")
    protocol = {"schema_version": 3, "forecast_model_version": forecast_version, "forecast_sha256": forecast_sha256,
                "reference_scope": "original 2025 training inputs", "reference_rows": 200, "latest_rows": "all", "background_rows": 100,
                "maximum_batch_rows": 128, "maximum_dense_cells": 2_000_000,
                "seed": 42, "family": family, "units": units, "features": FEATURES,
                "shap_version": shap_version, "method": "linear independent background or exact tree-path TreeSHAP; native sparse XGBoost"}
    upstream_hashes = {str(path.relative_to(cfg.PROJECT_ROOT)): file_hash(path) for path in main_paths + anomaly_paths}
    upstream_hashes.update({f"prepared/{name}": value for name, value in input_hashes.items()})
    # Isolate each calculation/input version, so rebuilding never writes into a valid active export.
    notebook = json.loads((Path(cfg.__file__).parent / "12_shap_and_dashboard_inputs.ipynb").read_text())
    code_text = "\n".join("".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code")
    protocol["calculation_sha256"] = hashlib.sha256(code_text.encode()).hexdigest()
    export_id = hashlib.sha256(json.dumps({"protocol":protocol, "upstream":upstream_hashes}, sort_keys=True).encode()).hexdigest()[:16]
    DASHBOARD_DIR = DASHBOARD_DIR / export_id
    PUBLISHED_MANIFEST = DASHBOARD_DIR / "dashboard_manifest.json"
    OUTPUT_DIR = DASHBOARD_DIR / "cache"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_FILES = ["latest_dashboard.parquet", "shap_explanations.parquet", "reference_feature_importance.csv"]
    cache_path = OUTPUT_DIR / "explanation_manifest.json"
    cache_manifest = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    reuse_cache = (cache_manifest.get("protocol") == protocol and cache_manifest.get("upstream_sha256") == upstream_hashes
                   and set(cache_manifest.get("output_sha256", {})) == set(CACHE_FILES)
                   and all((OUTPUT_DIR / name).is_file() and file_hash(OUTPUT_DIR / name) == cache_manifest["output_sha256"][name]
                           for name in CACHE_FILES))

In [7]:
if export_mode == "original_export":
    if reuse_cache:
        print("Reusing the verified explanation cache; no SHAP computation needed.")
    else:
        print("Building/regenerating only this deployment cache. Earlier files remain unchanged.")
        started = perf_counter()
        # Read training inputs only; no training/evaluation labels or test values are loaded.
        training = pd.read_parquet(PREPARED_DIR / "train.parquet", columns=COLUMNS)
        if training.empty or not training.next_week_end.le(pd.Timestamp(settings["train_end"])).all():
            raise ValueError("Training reference crosses its saved cutoff or is empty.")
        reference = training.sample(min(protocol["reference_rows"], len(training)), random_state=protocol["seed"])
        background = training.sample(min(protocol["background_rows"], len(training)), random_state=protocol["seed"])
        latest_sample = dashboard
        explanation_parts = []
        explainer = None
        for cohort, frame in [("historical_reference", reference), ("latest", latest_sample)]:
            values, base, raw, scores, explainer = explain_rows(forecast_model, family, frame, background, settings, explainer)
            part = frame[IDENTIFIERS].reset_index(drop=True).copy()
            part["cohort"] = cohort
            part["explanation_id"] = frame.forecast_id.to_numpy() if cohort == "latest" else [f"training_{i:05d}" for i in range(len(frame))]
            part["base_value"], part["raw_output"], part["model_score"] = base, raw, scores
            part[["shap__" + name for name in FEATURES]] = values
            explanation_parts.append(part)
        explanations = pd.concat(explanation_parts, ignore_index=True)
        reference_shap = explanations.loc[explanations.cohort.eq("historical_reference"), ["shap__" + name for name in FEATURES]]
        importance = pd.DataFrame({"feature": FEATURES, "mean_abs_shap": reference_shap.abs().mean().to_numpy(),
                                   "reference_rows": len(reference), "units": units}).sort_values("mean_abs_shap", ascending=False)
        explained_ids = set(explanations.loc[explanations.cohort.eq("latest"), "explanation_id"])
        dashboard["has_cached_explanation"] = dashboard.forecast_id.isin(explained_ids)
        dashboard.to_parquet(OUTPUT_DIR / CACHE_FILES[0], index=False)
        explanations.to_parquet(OUTPUT_DIR / CACHE_FILES[1], index=False)
        importance.to_csv(OUTPUT_DIR / CACHE_FILES[2], index=False)
        cache_manifest = {"protocol": protocol, "upstream_sha256": upstream_hashes,
            "output_sha256": {name: file_hash(OUTPUT_DIR / name) for name in CACHE_FILES},
            "selected_candidate": pin["candidate"], "forecast_model": forecast_record,
            "anomaly_reference": "January–December 2025; separate detector",
            "frozen_score_cutoff": selection["score_cutoff"],
            "latest_complete_week": str(expected_week.date()), "latest_supported_rows": len(dashboard),
            "latest_explained_rows": len(latest_sample), "training_reference_rows": len(reference),
            "offline_cache_build_seconds": perf_counter() - started,
            "package_versions": {name: metadata.version(name) for name in ["shap", "numpy", "pandas", "scikit-learn", "joblib"]},
            "limitations": ["Uncalibrated forecasting scores", "No causal attribution", "Importance uses a common 2025 reference sample, not the entire longer-history training population", "All supported latest rows explained; historical reference remains sampled", "Historical supported-row snapshot; no measured affordability benefit"]}
        temporary = cache_path.with_suffix(".json.tmp")
        temporary.write_text(json.dumps(cache_manifest, indent=2))
        temporary.replace(cache_path)

## 4. Verify explanations, then publish the dashboard handoff
All SHAP contributions must reconstruct the chosen model's output. The final stable manifest is written only after these checks pass. It identifies the forecasting checkpoint, frozen selected cutoff, evidence period/status, separate anomaly reference, file hashes and historical snapshot week.

The local file-read timing below is not a measurement of dashboard/browser responsiveness.

In [8]:
if export_mode == "original_export":
    read_started = perf_counter()
    dashboard = pd.read_parquet(OUTPUT_DIR / CACHE_FILES[0])
    explanations = pd.read_parquet(OUTPUT_DIR / CACHE_FILES[1])
    importance = pd.read_csv(OUTPUT_DIR / CACHE_FILES[2])
    cache_read_ms = 1000 * (perf_counter() - read_started)
    if not np.isfinite(dashboard.model_score.to_numpy()).all() or not np.isfinite(dashboard.anomaly_score.to_numpy()).all():
        raise ValueError("Non-finite dashboard scores; nothing published.")
    SHAP_COLUMNS = ["shap__" + name for name in FEATURES]
    if not np.isfinite(explanations[SHAP_COLUMNS + ["base_value", "raw_output", "model_score"]].to_numpy(dtype=float)).all():
        raise ValueError("Non-finite explanation; nothing published.")
    np.testing.assert_allclose(explanations.base_value + explanations[SHAP_COLUMNS].sum(axis=1), explanations.raw_output, rtol=1e-5, atol=1e-5)
    linked_scores = explanations.raw_output if family == "Random Forest" else expit(explanations.raw_output)
    np.testing.assert_allclose(linked_scores, explanations.model_score, rtol=1e-5, atol=1e-6)
    latest_explanations = explanations.loc[explanations.cohort.eq("latest")].set_index("explanation_id")
    assert dashboard.forecast_id.is_unique and explanations.explanation_id.is_unique
    if set(latest_explanations.index) != set(dashboard.forecast_id) or len(latest_explanations) != len(dashboard):
        raise ValueError("Full latest SHAP coverage is required before publishing.")
    assert dashboard.has_cached_explanation.eq(dashboard.forecast_id.isin(latest_explanations.index)).all()
    np.testing.assert_allclose(dashboard.set_index("forecast_id").loc[latest_explanations.index, "model_score"], latest_explanations.model_score, rtol=1e-6, atol=1e-8)
    print("Verified additive explanations. No model was fitted or reselected.")
    print(f"Cached latest explanations: {len(latest_explanations):,} / {len(dashboard):,} supported rows")
    print(f"Original offline build: {cache_manifest['offline_cache_build_seconds']:.2f} s | Current three-file read: {cache_read_ms:.2f} ms")
    print("File-read timing is not a measurement of dashboard/browser responsiveness.")
    display(dashboard[IDENTIFIERS + ["model_score", "predicted_surge", "anomaly_score", "is_anomaly", "has_cached_explanation"]].head(10))
    display(importance)
    fig, ax = plt.subplots(figsize=(9, 4))
    importance.head(12).iloc[::-1].plot.barh(x="feature", y="mean_abs_shap", legend=False, ax=ax)
    ax.set(xlabel=f"Mean absolute contribution ({units})", title="Descriptive importance: fixed 2025 reference inputs")
    fig.tight_layout(); plt.show()
    if len(latest_explanations):
        case = latest_explanations.iloc[0]
        contributions = pd.Series(case[SHAP_COLUMNS].to_numpy(dtype=float), index=FEATURES)
        chosen = contributions.abs().nlargest(8).index
        shown = contributions.loc[chosen].copy()
        shown.loc["All other features"] = contributions.drop(chosen).sum()
        shown = shown.iloc[::-1]
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.barh(shown.index, shown.values, color=np.where(shown.values >= 0, "tab:red", "tab:blue"))
        ax.axvline(0, color="black", linewidth=.7)
        ax.set(xlabel=f"Contribution ({units}); red increases the model score", title=f"Latest example: {case.name}")
        fig.tight_layout(); plt.show()
        print(f"Baseline {case.base_value:.4f} + all contributions = raw output {case.raw_output:.4f}; model score {case.model_score:.4f}.")
    else:
        print("No supported latest rows/explanations. No older forecasts were substituted.")
    # Recheck all score inputs before publishing the stable entry point.
    for path in main_paths + anomaly_paths:
        if file_hash(path) != upstream_hashes[str(path.relative_to(cfg.PROJECT_ROOT))]:
            raise ValueError("A model or anomaly input changed during cache building.")
    for relative, expected in input_hashes.items():
        experiment, filename = relative.split("/", 1)
        if file_hash(cfg.PROJECT_ROOT / "experiments" / experiment / "05_modelling" / filename) != expected:
            raise ValueError("Prepared display inputs changed during export.")
    for relative, expected in refresh_inputs.items():
        if file_hash(cfg.PROJECT_ROOT / relative) != expected:
            raise ValueError("Refresh context changed during export.")
    # Publish a small stable entry point only after cache integrity/additivity checks pass.
    published = {"schema_version": 1, "cache_dir": OUTPUT_DIR.name,
        "cache_manifest_sha256": file_hash(cache_path), "forecast_model": forecast_record,
        "anomaly_model": {"sha256": anomaly_manifest["model_sha256"],
                          "manifest_path": str(anomaly_paths[0].relative_to(cfg.PROJECT_ROOT)), "reference_period": "January–December 2025",
                          "interpretation": "separate unusual-pattern signal; no anomaly ground truth"},
        "latest_complete_week": str(expected_week.date()), "latest_supported_rows": len(dashboard),
        "files": {name: str(Path(OUTPUT_DIR.name) / name) for name in CACHE_FILES},
        "limitations": cache_manifest["limitations"]}
    temporary = PUBLISHED_MANIFEST.with_suffix(".tmp")
    temporary.write_text(json.dumps(published, indent=2))
    temporary.replace(PUBLISHED_MANIFEST)
    print("Dashboard entry point:", PUBLISHED_MANIFEST)
    print("Forecast model:", forecast_record["version"], "|", forecast_record["evidence_status"])
    print("Saved verified dashboard inputs and explanations to:", OUTPUT_DIR)

    # Verify the pin and source study again; publish the active pointer LAST.
    cfg.dashboard_candidate()
    active = cfg.PROJECT_ROOT / "08_dashboard" / "active_deployment.json"
    pointer = {"schema_version": 1, "deployment_id": pin["deployment_id"],
               "directory": str(DASHBOARD_DIR.relative_to(active.parent))}
    temporary = active.with_suffix(".tmp")
    temporary.write_text(json.dumps(pointer, indent=2))
    temporary.replace(active)
    print("Activated verified candidate-specific dashboard. Legacy exports were preserved.")

## Handoff and report wording

The app follows `08_dashboard/active_deployment.json` only after all checks pass. Its model-specific export contains the selected classifier, frozen cutoff and its own validation/group results. Old root exports and refreshes are preserved for rollback; a failed build does not activate a partial cache.

The anomaly signal remains a separate 2025 reference. SHAP is associative, not causal, and its global summary uses the stated 2025 reference sample. Missing explanations are explicit. Candidate metrics reuse validation cutoff selection; do not report another model's original test results as this candidate's performance. Use 13 to refresh observations without fitting or changing thresholds.

## Verify the saved-snapshot route

When `auto` selected an existing snapshot, the original-export cells above were skipped. This step expands its local explanations if needed, or verifies and reuses full coverage. It prints the actual week, coverage and cache location. No main 13 rerun is needed just to explain the saved snapshot. Main 13 is only for newer observations, subject to its registration guard.

In [9]:
if export_mode == "saved_snapshot":
    import app
    entry_path = expand_shap_cache()
    entry, manifest, latest, explanations, _, _ = app.load_exports(
        str(cfg.PROJECT_ROOT), app.export_signature(cfg.PROJECT_ROOT))
    local = explanations.loc[explanations.cohort.eq("latest")]
    if len(local) != len(latest) or set(local.explanation_id) != set(latest.forecast_id):
        raise ValueError("Not every supported latest row has an explanation.")
    print("Observed week:", entry["latest_complete_week"])
    print(f"Verified latest SHAP coverage: {len(local):,} / {len(latest):,}")
    print("Dashboard entry:", entry_path)
    print("Same prices, model, scores and cutoff. Restart Streamlit to load the explanations.")


2026-11-16 18:20:09.586 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Explained 128 / 5,357 rows
Explained 256 / 5,357 rows
Explained 384 / 5,357 rows
Explained 512 / 5,357 rows
Explained 640 / 5,357 rows
Explained 768 / 5,357 rows
Explained 896 / 5,357 rows
Explained 1,024 / 5,357 rows
Explained 1,152 / 5,357 rows
Explained 1,280 / 5,357 rows
Explained 1,408 / 5,357 rows
Explained 1,536 / 5,357 rows
Explained 1,664 / 5,357 rows
Explained 1,792 / 5,357 rows
Explained 1,920 / 5,357 rows
Explained 2,048 / 5,357 rows
Explained 2,176 / 5,357 rows
Explained 2,304 / 5,357 rows
Explained 2,432 / 5,357 rows
Explained 2,560 / 5,357 rows
Explained 2,688 / 5,357 rows
Explained 2,816 / 5,357 rows
Explained 2,944 / 5,357 rows
Explained 3,072 / 5,357 rows
Explained 3,200 / 5,357 rows
Explained 3,328 / 5,357 rows
Explained 3,456 / 5,357 rows
Explained 3,584 / 5,357 rows
Explained 3,712 / 5,357 rows
Explained 3,840 / 5,357 rows
Explained 3,968 / 5,357 rows
Explained 4,096 / 5,357 rows
Explained 4,224 / 5,357 rows
Explained 4,352 / 5,357 rows
Explained 4,480 / 5,357 rows